# Figure 3 fill sensitivity

How many calls of each Figure 3 set are geNomad viral with a Metabolism KO, the dark-green fill of Figure 3, under the primary rule and under each sensitivity variant? The primary rule (`amg_shared_kegg.ipynb`) counts a geNomad-viral call whose shared KofamScan KO, the best hit at an e-value of at most 1e-5, sits anywhere under KEGG A 09100 Metabolism, and Figure 3 keeps it. The variants remove nucleotide metabolism KOs, cautious functions or below-threshold hits, or pool the high and medium levels, and were compared after the primary counts had been seen. They are reported in Source Data only.


## Inputs

`shared_ko_per_gene.parquet` of `amg_shared_kegg.ipynb` holds the shared KO and the four-class label of every gene called at any tier. `AMG_call_weight_caution.parquet` of `amg_benchmark_comparisons.ipynb` holds each tool's own cautious flag per call. `AMG_filters.tsv` is the CheckAMG cautious-function list and `ko00001.keg` the pinned KO hierarchy, `ko00001.keg` is the pinned KO hierarchy under `./data`.


In [ ]:
import os

os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
from pathlib import Path

TBL_DIR = Path("./tables/amg_benchmark")
SHARED_KO = TBL_DIR.joinpath("shared_ko_per_gene.parquet")
CALL_CAUTION = TBL_DIR.joinpath("AMG_call_weight_caution.parquet")
AMG_FILTERS = Path("../CheckAMG/files/AMG_filters.tsv")
KO00001 = Path("./data/ko00001.keg")


## Method

### BRITE B nodes under Metabolism

Each KO of `ko00001.keg` is listed with the set of B nodes under A 09100 that hold it. The strict nucleotide variant removes a dark-green call whose KO sits under B 09104 (Nucleotide metabolism) at all, even when the KO also sits under another 09100 B node. The lenient variant removes it only when 09104 is its only 09100 B node.


In [ ]:
metabolism_b_nodes = {}
a_node = b_node = None
for line in open(KO00001, encoding="utf-8", errors="replace"):
    if line.startswith("A"):
        a_node = line[1:6]
    elif line.startswith("B"):
        fields = line[1:].split()
        b_node = fields[0] if fields and fields[0].isdigit() else None
    elif line.startswith("D") and a_node == "09100" and b_node:
        fields = line[1:].split()
        if fields and fields[0].startswith("K"):
            metabolism_b_nodes.setdefault(fields[0], set()).add(b_node)
ko_b_nodes = pl.DataFrame({"S_ko": list(metabolism_b_nodes), "metabolism_B": [sorted(v) for v in metabolism_b_nodes.values()]})
print("KOs under 09100", ko_b_nodes.height, "| 09100 B nodes", sorted({b for v in metabolism_b_nodes.values() for b in v}))
print("KOs under 09104", sum("09104" in v for v in metabolism_b_nodes.values()),
      "| under 09104 alone", sum(v == {"09104"} for v in metabolism_b_nodes.values()))

ko00001 places 6,490 KOs under A 09100, in the twelve B nodes 09101 to 09112. 299 of them sit under B 09104 (Nucleotide metabolism), and 212 of those under no other 09100 B node, so the strict variant removes more calls than the lenient one only where a call's KO is shared between 09104 and another Metabolism node.


### Cautious flags

The CheckAMG cautious list is `AMG_filters.tsv`. Its rows are grouped by database and identifier, with Pfam identifiers cut at the version suffix, and a KO is cautious when any of the five flags `filter_essential`, `filter_glucan`, `filter_lipid`, `filter_methyl` and `filter_nucleotide` is set on any of its KEGG rows. The shared KO of each call is looked up in that list. The tool-own variant instead uses the `cautious` flag that each calling tool's own annotation received in `AMG_call_weight_caution.parquet`, which is constant across the tiers of one tool and gene, and a call with no row for that tool counts as not cautious.


In [ ]:
FILTER_FLAGS = ["filter_essential", "filter_glucan", "filter_lipid", "filter_methyl", "filter_nucleotide"]
filters = pl.read_csv(AMG_FILTERS, separator="\t", infer_schema_length=0)
cautious_kos = (
    filters.with_columns([(pl.col(flag) == "True").alias(flag) for flag in FILTER_FLAGS])
    .with_columns(pl.when(pl.col("db") == "Pfam").then(pl.col("id").str.split(".").list.first()).otherwise(pl.col("id")).alias("id_key"))
    .group_by("db", "id_key").agg([pl.col(flag).any() for flag in FILTER_FLAGS])
    .filter(pl.col("db") == "KEGG")
    .select(pl.col("id_key").alias("S_ko"), pl.any_horizontal(FILTER_FLAGS).alias("ko_cautious"))
)
call_caution = pl.read_parquet(CALL_CAUTION, columns=["tool", "sample", "gene", "cautious"]).unique()
assert not call_caution.select("tool", "sample", "gene").is_duplicated().any(), "a tool's cautious flag differs across tiers"
TOOL_CAUTION = {"CheckAMG": "cautious_CheckAMG", "DRAM-V": "cautious_DRAMV", "VIBRANT": "cautious_VIBRANT"}
genes = (
    pl.read_parquet(SHARED_KO).join(ko_b_nodes, on="S_ko", how="left").join(cautious_kos, on="S_ko", how="left")
    .with_columns(pl.col("ko_cautious").fill_null(False),
                  pl.col("metabolism_B").list.contains("09104").fill_null(False).alias("under_09104"),
                  (pl.col("metabolism_B").list.len() == 1).fill_null(False).alias("single_metabolism_B"))
)
for tool, column in TOOL_CAUTION.items():
    genes = genes.join(call_caution.filter(pl.col("tool") == tool).select("sample", "gene", pl.col("cautious").alias(column)),
                       on=["sample", "gene"], how="left").with_columns(pl.col(column).fill_null(False))
print("KEGG KOs in AMG_filters.tsv", cautious_kos.height, "| cautious", int(cautious_kos["ko_cautious"].sum()))
print("genes", genes.height)

`AMG_filters.tsv` lists 9,891 KEGG KOs, and every one carries at least one of the five flags, so on the shared KO the cautious variant removes every dark-green call whose KO the list holds. All 306,534 genes of `shared_ko_per_gene.parquet` are kept.


### Dark-green calls per set under each rule

The sets are those of Figure 3: the nine tiers of panel A after the primary-tier filter (a gene called at CheckAMG high, DRAM-V default or VIBRANT high), whose DRAM-V rows `DRAMV_aux1`, `DRAMV_aux2` and `DRAMV_aux3` are the cumulative sets DRAM-V (aux 1), (aux 2) and (aux 3), the calls at auxiliary score 1, at most 2 and at most 3, the three focal tiers of panel B in full, and the nine cumulative matched levels in full, named as the Figure 3 sets (: level 1 CheckAMG (high), DRAM-V (aux 1) and VIBRANT (high), level 2 CheckAMG (medium), DRAM-V (aux 2) and VIBRANT (medium), level 3 CheckAMG (low), DRAM-V (aux 3) and VIBRANT (low), where a CheckAMG or VIBRANT level includes the tiers above it and DRAM-V (aux 3) is DRAM-V (default)). The three combined rows of earlier versions equaled the level 2 rows and are not written. Each row gives the calls, the geNomad-viral calls and, per rule, the dark-green calls and their share of all calls of the set. `adaptive_threshold` is the four-class label under the KofamScan adaptive-threshold rule (`P_S2_fill`).


In [ ]:
DARK_GREEN = "viral, KEGG Metabolism KO"
primary = pl.col("S_S2_fill") == DARK_GREEN
RULES = {
    "primary": primary,
    "nucleotide_excluded_strict": primary & ~pl.col("under_09104"),
    "nucleotide_excluded_lenient": primary & ~(pl.col("under_09104") & pl.col("single_metabolism_B")),
    "cautious_shared_ko": primary & ~pl.col("ko_cautious"),
    "cautious_tool_own": None,
    "adaptive_threshold": pl.col("P_S2_fill") == DARK_GREEN,
}
filtered = pl.col("CheckAMG_high") | pl.col("DRAMV_default") | pl.col("VIBRANT_high")
PANEL_A_CUMULATIVE = {"DRAMV_aux2": pl.col("DRAMV_aux1") | pl.col("DRAMV_aux2"), "DRAMV_aux3": pl.col("DRAMV_default")}
SETS = [("A, primary-tier filter", tier, filtered & PANEL_A_CUMULATIVE.get(tier, pl.col(tier))) for tier in
        ["CheckAMG_high", "CheckAMG_medium", "CheckAMG_low", "DRAMV_aux1", "DRAMV_aux2", "DRAMV_aux3", "VIBRANT_high", "VIBRANT_medium", "VIBRANT_low"]]
SETS += [("B focal, full tier", tier, pl.col(tier)) for tier in ["CheckAMG_high", "DRAMV_default", "VIBRANT_high"]]
MATCHED = {
    "CheckAMG (high)": pl.col("CheckAMG_high"), "DRAM-V (aux 1)": pl.col("DRAMV_aux1"), "VIBRANT (high)": pl.col("VIBRANT_high"),
    "CheckAMG (medium)": pl.col("CheckAMG_high") | pl.col("CheckAMG_medium"), "DRAM-V (aux 2)": pl.col("DRAMV_aux1") | pl.col("DRAMV_aux2"),
    "VIBRANT (medium)": pl.col("VIBRANT_high") | pl.col("VIBRANT_medium"),
    "CheckAMG (low)": pl.col("CheckAMG_high") | pl.col("CheckAMG_medium") | pl.col("CheckAMG_low"),
    "DRAM-V (aux 3)": pl.col("DRAMV_default"), "VIBRANT (low)": pl.col("VIBRANT_high") | pl.col("VIBRANT_medium") | pl.col("VIBRANT_low"),
}
SETS += [("matched level, full tier", level, keep) for level, keep in MATCHED.items()]


def tool_of(name):
    return "CheckAMG" if name.startswith("CheckAMG") else ("VIBRANT" if name.startswith("VIBRANT") else "DRAM-V")


rows = []
for panel, name, keep in SETS:
    calls = genes.filter(keep)
    n_calls = calls.height
    row = {"panel": panel, "set": name, "tool": tool_of(name), "n_calls": n_calls, "n_genomad_viral": int(calls["genomad_viral"].sum())}
    for rule, flag in RULES.items():
        if flag is None:
            flag = primary & ~pl.col(TOOL_CAUTION[tool_of(name)])
        row[rule] = calls.select(flag.sum()).item()
        row[f"{rule}_share"] = row[rule] / n_calls
    rows.append(row)
sensitivity = pl.DataFrame(rows)
SENSITIVITY_PATH = TBL_DIR.joinpath("fig3_fill_sensitivity.tsv")
sensitivity.write_csv(SENSITIVITY_PATH, separator="\t")
with pl.Config(tbl_rows=30, tbl_cols=20, tbl_width_chars=400, fmt_str_lengths=40, ascii_tables=True):
    print(sensitivity.select(pl.exclude("^.*_share$", "tool")))

At the first matched level CheckAMG (high) holds 3,221 calls, 3,153 of them geNomad viral, and 1,751 dark-green calls under the primary rule, which become 1,736 with nucleotide metabolism excluded strictly, 1,751 leniently, 582 with the CheckAMG cautious list on the shared KO, 649 with CheckAMG's own cautious flag and 841 under the adaptive threshold. DRAM-V at auxiliary score 1 holds 3,380 calls, 3,135 geNomad viral, and 1,637 dark-green calls, which become 1,285, 1,421, 216, 667 and 723. VIBRANT (high) holds 1,270 calls, 1,263 geNomad viral, and 1,161 dark-green calls, which become 1,134, 1,156, 190, 139 and 534. Strict nucleotide exclusion changes DRAM-V most, from 1,637 to 1,285 dark-green calls at aux 1, against 1,751 to 1,736 for CheckAMG (high) and 1,161 to 1,134 for VIBRANT (high). At the cumulative second level CheckAMG (medium) holds 6,728 calls, 5,843 geNomad viral and 3,475 dark-green, DRAM-V (aux 2) holds 13,566, 9,490 and 5,404, and VIBRANT (medium) 3,720, 3,708 and 3,442. At the third level CheckAMG (low) holds 170,161 calls, 8,879 geNomad viral and 5,456 dark-green, DRAM-V (aux 3) holds 47,308, 16,746 and 10,244, and VIBRANT (low) 12,677, 12,531 and 11,599. DRAM-V (default) in full, the focal set of panel B, holds 47,308 calls and 10,244 dark-green calls under the primary rule.


## Outputs

`tables/amg_benchmark/fig3_fill_sensitivity.tsv`, the calls, geNomad-viral calls and dark-green calls of every Figure 3 set under the primary rule and each sensitivity variant. `amg_benchmark_figures.Rmd` reads it and writes the Source Data CSV.
